In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('/home/jovyan/kuratov/data/test_time_gd/')

In [2]:
from kv_dataset_utils import generate_sequence

In [24]:
num_kv_pairs = 10
k_length = 4
v_length = 4
n_segments = 4
min_segment_len = 32
max_segment_len = 64

In [25]:
print('KV pairs length:', (k_length + v_length + 3)*num_kv_pairs)
print('~Total length:', n_segments * (min_segment_len + max_segment_len)/2)

KV pairs length: 110
~Total length: 192.0


In [18]:
sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments, min_segment_len, max_segment_len)
sample

{'kv_pairs': ['!fL:jA!',
  '!jt:y6!',
  '!eD:yR!',
  '!Qa:lU!',
  '!xl:u9!',
  '!Rl:t1!',
  '!X6:TO!',
  '!12:g7!',
  '!er:zd!',
  '!C4:hi!'],
 'segment_ids_to_kv_ids': {0: [0, 4, 6, 9], 1: [1, 5, 7], 2: [], 3: [2, 3, 8]},
 'context': 'uET!C4:hi!4!X6:TO!rh!xl:u9!!fL:jA!|2!12:g7!EZj!Rl:t1!iV7RvTb503qlwQhDlEdJBeyiNaQK6!jt:y6!z8Esc3|ByoBueehfjT3F9I274UvlzIns0dvIRs0lXu7QKf2gaa|!er:zd!UFbbNKcI!Qa:lU!1LZ!eD:yR!MT0CCHd3|',
 'query': '?!fL:',
 'input_sequence': 'uET!C4:hi!4!X6:TO!rh!xl:u9!!fL:jA!|2!12:g7!EZj!Rl:t1!iV7RvTb503qlwQhDlEdJBeyiNaQK6!jt:y6!z8Esc3|ByoBueehfjT3F9I274UvlzIns0dvIRs0lXu7QKf2gaa|!er:zd!UFbbNKcI!Qa:lU!1LZ!eD:yR!MT0CCHd3|?!fL:',
 'target': 'jA!|'}

In [19]:
from datasets import Dataset, DatasetDict
from tqdm import tqdm

num_samples = 1_000_000

data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments, min_segment_len, max_segment_len)
    data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['target'],
    }]
data = Dataset.from_list(data)

num_samples = 5_000

valid_data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments, min_segment_len, max_segment_len)
    valid_data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['target'],
    }]
valid_data = Dataset.from_list(valid_data)

dataset = DatasetDict({'train': data, 'valid': valid_data})

100%|██████████| 5000/5000 [00:00<00:00, 14716.96it/s]


In [20]:
dataset

DatasetDict({
    train: Dataset({
        features: ['context', 'query', 'target'],
        num_rows: 1000000
    })
    valid: Dataset({
        features: ['context', 'query', 'target'],
        num_rows: 5000
    })
})

In [21]:
dataset_name = f'N{num_kv_pairs}-K{k_length}V{v_length}-S{n_segments}({min_segment_len}-{max_segment_len})_1M'
dataset.save_to_disk(f'./data/{dataset_name}')

Saving the dataset (0/1 shards):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

# copy task dataset

In [5]:
from kv_dataset_utils import generate_sequence, ALPHABET

In [12]:
from kv_dataset_utils import generate_sequence

num_kv_pairs = 0
k_length = 4
v_length = 4
n_segments = 1
min_segment_len = 4
max_segment_len = 4

sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                           min_segment_len, max_segment_len)
sample

{'kv_pairs': [],
 'segment_ids_to_kv_ids': {0: []},
 'context': 'v42v|',
 'query': '?!:',
 'input_sequence': 'v42v|?!:',
 'target': ''}

In [13]:
from datasets import Dataset, DatasetDict
from tqdm import tqdm

num_samples = 1_000_000

data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments, min_segment_len, max_segment_len)
    data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['context'],
    }]
data = Dataset.from_list(data)

num_samples = 5_000

valid_data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments, min_segment_len, max_segment_len)
    valid_data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['context'],
    }]
valid_data = Dataset.from_list(valid_data)

dataset = DatasetDict({'train': data, 'valid': valid_data})

100%|██████████| 5000/5000 [00:00<00:00, 170832.10it/s]


In [14]:
dataset_name = f'N{num_kv_pairs}-S{n_segments}({min_segment_len}-{max_segment_len})_1M'
dataset.save_to_disk(f'./data/{dataset_name}')

Saving the dataset (0/1 shards):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [15]:
dataset = dataset.load_from_disk(f'./data/{dataset_name}')